# Paper 4 — Early Detection of Food Price Shocks with Cross-Product Temporal Attention

**Research question.** Do signals from *other* products — co-moving categories, platform-wide repricing,
recent increases among similar goods — give earlier and more reliable warning of an upcoming price shock than
a product's own history?

**Target.** For product i at origin t and horizon h in {1, 7, 14}:

shock(i, t, h) = 1  if  100 x log(P[t+h] / P[t]) >= theta(c, h),

where theta(c, h) = max(5, median of positive h-day log-changes in category c during the *training* window).
Thresholds are re-estimated per fold on training data only (no test leakage). Scraper-outage targets are
masked.

**Models.** Persistence heuristics; logistic regression and LightGBM on own-history + cross-product
features; a single-task temporal transformer (own history only); and **MT-CPTA (proposed)**: a patch
temporal encoder per product, cross-product attention across all products at the same date, and three
heads trained jointly — price path (Huber), shock (focal loss) and forward volatility (MSE):
L = L_forecast + L_shock + 0.5 L_volatility.

**Evaluation.** PR-AUC (primary; shocks are rare), ROC-AUC, Brier score, expected calibration error (raw and
Platt-calibrated on validation), and F1 / precision / recall at the validation-optimal threshold; product-level
cluster-bootstrap CIs for PR-AUC differences; lead-time analysis of detected shock events; event study around
platform batch-update days.

**Outputs** under `food_price_research/paper4_shock_detection/`.

## 2. Dataset description

In [ ]:
# ==== 0. CONFIGURATION & GOOGLE DRIVE =====================================
PAPER = "paper4_shock_detection"
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")
SMOKE = os.environ.get("FPR_SMOKE", "0") == "1"   # True -> tiny, fast end-to-end test run
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/food_price_research"
else:
    ROOT = os.environ.get("FPR_ROOT", os.path.abspath("food_price_research"))
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'lightgbm'], check=False)

from pathlib import Path
ROOT = Path(ROOT); RAW = ROOT / "raw"; RAW.mkdir(parents=True, exist_ok=True)
PREP_DIR = ROOT / "shared_prep"                      # cleaned panel, shared by all five papers
PAPER_DIR = ROOT / PAPER
FIG_DIR, TAB_DIR, RES_DIR, CKPT_DIR = [PAPER_DIR / s for s in ("figures", "tables", "results", "checkpoints")]
for d in (PREP_DIR, FIG_DIR, TAB_DIR, RES_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

cands = sorted(RAW.glob("[dD]ataset-2024.*"))
if not cands and IN_COLAB:
    from google.colab import files
    print("Upload dataset-2024.csv (or Dataset-2024.xlsx) - it is saved to", RAW)
    for k, v in files.upload().items():
        (RAW / k).write_bytes(v)
    cands = sorted(RAW.glob("[dD]ataset-2024.*"))
assert cands, f"Put dataset-2024.csv in {RAW}"
DATA_PATH = cands[0]
print("paper:", PAPER, "| data:", DATA_PATH, "| smoke:", SMOKE)

## 1. Shared data preparation (identical in all five notebooks)

The raw panel is a wide table (`Name | 2024-01-02 | ... | 2025-02-05`): 5,976 SKUs x 401 days, already
complete. Quality control, all estimated on training data only:

* **Scraper outages.** On 56 days not a single SKU changes price (the median day has ~80 changes): the scraper
  did not run and prices were carried forward. These cells are *imputed*, so they are excluded from every loss,
  metric target and graph statistic. The first day after an outage (a *gap-end* day) accumulates several days of
  changes and is excluded from co-movement statistics.
* **Spike-and-revert repair.** A jump of more than x1.5 that returns to within 5 % of the previous level within
  7 days is treated as a scrape error (wrong variant shown) and replaced by the previous level.
* **Variant breaks.** A persistent level shift of more than x3 within the training window is almost surely a
  re-used product slot; those SKUs are excluded.
* **Platform batch-update days.** Days whose number of simultaneous changes exceeds median + 5 MAD. Batch
  updates create spurious co-movement, so they are excluded from correlation-type statistics.
* **Category tagger.** Priority-ordered keyword rules on product names; non-food SKUs are kept as a control
  group and excluded from headline results.
* **Activity tiers** (on the earliest training window, so no look-ahead): T0 food SKUs that never change,
  T1 1-9 changes, T2 >= 10 changes. Headline results use the *active* universe T1 u T2.
* **Rolling-origin evaluation.** Three folds, each with a 30-day validation and a 30-day test window
  (test windows: 2024-11-08..12-07, 12-08..01-06, 2025-01-07..02-05). Models never see validation or test
  targets during fitting; forecasts at each origin use only data up to that origin.
* **Targets.** 100 x log(P[t+h] / P[t]) for h = 1..30; metrics are computed in price space at h in {1,3,7,14,30}.
  Because most prices are sticky we also report **CC-sMAPE** (sMAPE only where the price actually changes) and
  **change AUPRC**, alongside MAE, RMSE, MAPE, sMAPE, WAPE and MASE.

In [ ]:
# ============================================================================
# SECTION 1 — SHARED DATA PREPARATION (identical, deterministic in every notebook)
# ============================================================================
import re, json, time, math, pickle, hashlib, warnings, itertools
from pathlib import Path
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# 1.1  Category tagger (rule-based, from product names; priority-ordered)
# ---------------------------------------------------------------------------
# Non-food rules are checked FIRST so that e.g. "hair oil" never becomes edible
# oil and "baby shampoo" never becomes baby food.
CATEGORY_RULES = [
    # ---- non-food -------------------------------------------------------
    ("NF: Personal Care", r"shampoo|conditioner|face ?wash|facewash|facial|soap|lotion|deodorant|deo |roll on|perfume|body spray|"
                          r"toothpaste|toothbrush|tooth ?brush|mouthwash|hair oil|hair color|hair colour|hair removal|hair treatment|"
                          r"serum|sunscreen|sun protect|talcum|cream(?!.*(biscuit|wafer|cake|cheese|ice cream|milk|roll|sandwich))|"
                          r"vaseline|petroleum jelly|nose strip|shaving|razor|blade|sanitary|napkin|pad[s]? |tampon|condom|lip ?balm|"
                          r"lipstick|nail|kajal|mascara|glycerine|cleansing|scrub|makeup|cosmetic|beauty|skin|aloe ?vera gel|"
                          r"hand ?wash|body wash|shower gel|cotton bud|wipes|essential oil|facial oil|massage oil|coconut oil hair|"
                          r"boroplus|moisturi|toner|face mask|mask|blackhead|dandruff|gatsby|gloss|hair gel|lip colou?r|hair|shave|sanitizer|baby wash|baby bath|\blip"),
    ("NF: Baby Care",     r"diaper|nappy|pant [smlx]|belt \d|feeding bottle|bottles cleanser|nipple|baby pillow|nima|teether|"
                          r"baby lotion|baby oil|baby soap|baby shampoo|baby powder|baby wipes|pacifier|romper|cot sheet|pants .*kg"),
    ("NF: Household",     r"clean(er|ing)|detergent|dish ?wash|washing|bleach|disinfect|toilet|harpic|lizol|air freshener|"
                          r"freshener|aerosol|mosquito|insect|coil|killer|repellent|tissue|napkin|foil|cling film|garbage|"
                          r"broom|mop|brush|sponge|scrubber|battery|bulb|light|candle|matchbox|match box|lighter|polish|"
                          r"glass cleaner|floor|mat |mat$|pillow|towel|bucket|container|organizer|box$|jar |bottle(?!.*(juice|drink|water|milk|sauce|oil))|"
                          r"plate|bowl|spoon|glass |mug|cup(?! noodles)|tray|chopping board|knife|rolling pin|leveler|grill|net$|pan |pot |kettle|"
                          r"flask|tiffin|lunch box|hanger|umbrella|finish household|powerball|fabric|softener|surf excel|rin |wheel |"
                          r"cat food|dog food|pet |\bcat\b|kitten|\bdog\b|smartheart|pro diet|whiskas|pedigree|kitty|wrap|mixer|apron|strainer|duster|dish cover|holder|litter|"
                          r"spray|cockroach|\bhit\b|diesel|\bmobil\b|lubricant|\bcover\b|sheet"),
    ("NF: Health",        r"tablet|capsule|syrup(?!.*(chocolate|strawberry))|vitamin|antiseptic|dettol liquid|savlon|bandage|"
                          r"plaster|thermometer|oral saline|ors|pain relief|balm|inhaler|medicine|n95|surgical|glucose(?! biscuit)|"
                          r"test kit|pregnancy"),
    ("NF: Stationery & Misc", r"pencil|pen |pen$|ball ?pen|marker|highlighter|eraser|sharpener|notebook|note ?book|file |folder|"
                          r"stapler|punch|scissor|cutter|glue|tape|calculator|compass|geometry|ruler|a4|paper|envelope|sticker|"
                          r"flag|toy|puzzle|ball |badminton|shuttlecock|cricket|football|game|doll|remote|charger|cable|"
                          r"earphone|headphone|usb|mobile|phone|fusion set|moistener|clip|board|diary|book|colour pencil|crayon|"
                          r"gift|decoration|balloon|travel|bag|wallet|\bcard\b|\bbear\b|\bcar\b|instrument|acrylic|glitter|sash|tasbih|"
                          r"\bbat\b|lace|drawing|flower"),
    # ---- food ---------------------------------------------------------------
    ("Baby Food",         r"formula|infant|baby food|cerelac|nan optipro|lactogen|biomil|growing up milk|\(0-6|\(6-12|\(1-3|"
                          r"nido 1|nido 3|cereal.*(tin|baby)|mother's smile"),
    ("Beverages",         r"juice|drink|beverage|electrolyte|soda|cola|7 ?up|sprite|fanta|pepsi|coca|mojo|speed|energy|water|lassi|milk shake|"
                          r"shake|tang|horlicks|boost|ovaltine|bournvita|malt|squash|syrup|can$|smoothie|mocktail|aloe drink|frooto|frutika"),
    ("Tea & Coffee",      r"\btea\b|tea bag|green tea|coffee|nescafe|infusion|chai"),
    ("Dairy & Eggs",      r"milk|cheese|butter(?!.*(cookie|biscuit|peanut))|yogurt|yoghurt|\bdoi\b|curd|cream cheese|paneer|"
                          r"ghee|egg|dim\b|dairy|uht|evaporated|condensed|ice cream|kulfi"),
    ("Edible Oil",        r"soybean oil|soyabean|mustard oil|sunflower oil|palm oil|rice bran|olive oil|canola|"
                          r"vegetable oil|cooking oil|sesame oil|\boil\b"),
    ("Fish & Seafood",    r"fish|ilish|hilsha|rui|rohu|katla|pangas|tilapia|shing|magur|koi|pabda|chingri|shrimp|prawn|"
                          r"crab|squid|baila|boal|shol|taki|poa|rupchanda|pomfret|loitta|kachki|mola|tengra|shutki|dried fish"),
    ("Meat & Poultry",    r"beef|mutton|goat|lamb|chicken|broiler|deshi murgi|murgi|duck(?! eggs)|pigeon|quail|meat|"
                          r"liver|gizzard|keema|mince|bone in|boneless|drumstick|breast|wings|sausage|salami|nugget|patty|kebab"),
    ("Frozen & Ready",     r"frozen|ready to cook|ready mix|paratha|puri|samosa|singara|spring roll|haleem mix|biryani mix|"
                          r"instant|french fries|burger|pizza(?! sauce)"),
    ("Rice & Grains",     r"\brice\b|chinigura|basmati|miniket|nazirshail|atop|kalijira|polao|poha|chira|muri|khoi|oats|"
                          r"cornflakes|corn flakes|cereal|muesli|barley|quinoa|semai|shemai|laccha|lachcha|vermicelli|wheat"),
    ("Flour & Bakery",    r"atta|maida|flour|suji|semolina|besan|bread|bun|cake|rusk|toast|bakorkhani|bakery|dough|yeast|"
                          r"baking|pastry|croissant|donut|ruti|chapati|roti"),
    ("Lentils & Pulses",  r"\bdal\b|lentil|masur|moong|mung|mashkalai|chola|chick ?pea|kabuli|anchor dal|khesari|"
                          r"motor|peas|beans?\b|rajma"),
    ("Spices & Condiments", r"masala|spice|powder|chili|chilli|turmeric|holud|coriander|cumin|zeera|jeera|cardamom|elach|"
                          r"cinnamon|daruchini|clove|lobongo|black pepper|golmorich|bay leaf|tejpata|nutmeg|jayfal|mace|"
                          r"fenugreek|methi|panch phoron|mix$|salt|vinegar|flakes|seasoning|stock|soup|bouillon|gelatine|"
                          r"food colou?r|essence|baking soda|kewra|rose water|seed|herbs|china grass|mix\s*$"),
    ("Sugar & Sweeteners", r"sugar|chini|\bgur\b|jaggery|honey|mishri|sweetener|candy|chocolate|toffee|lollipop|jelly|"
                          r"marshmallow|gum|mint"),
    ("Snacks & Biscuits", r"biscuit|cookie|cracker|wafer|chips|chanachur|namkeen|dalmoth|bhujia|nuts|peanut(?! butter)|"
                          r"cashew|kaju|almond|pista|raisin|kismis|dates|khejur|popcorn|puffs|roll$|snack|bar |bar$|"
                          r"chanachur|jhal muri|sticks|crisps|pringles|kurkure|lays|badam"),
    ("Sauces & Spreads",  r"sauce|ketchup|mayonnaise|mayo|\bjam\b|jelly|spread|pickle|achar|chutney|peanut butter|nutella|"
                          r"dressing|paste|puree|olive(?! oil)"),
    ("Noodles & Pasta",   r"noodle|ramen|pasta|macaroni|spaghetti|fusilli|penne|maggi|cup noodles"),
    ("Fruits",            r"mango|aam|banana|kola\b|apple|orange|malta|grape|guava|peyara|papaya|pape|pineapple|anaros|"
                          r"watermelon|tormuj|lemon|lebu|lime|pomegranate|anar|litchi|lichu|jackfruit|kathal|coconut|"
                          r"narikel|dates|berry|kiwi|pear|plum|prune|wood apple|kodbel|amra|jambura|sofeda|fruit"),
    ("Vegetables",        r"potato|alu|aloo|onion|peyaj|piyaj|garlic|roshun|ginger|ada\b|tomato|brinjal|begun|cucumber|"
                          r"shosha|pumpkin|kumra|gourd|lau|korola|potol|dherosh|okra|cabbage|cauliflower|kopi|carrot|gajor|"
                          r"radish|mula|spinach|shak|sak\b|lettuce|capsicum|green chili|kacha morich|beans|borboti|"
                          r"mushroom|coriander leaves|dhonia pata|mint leaves|pudina|vegetable|sobji|corn|bhutta|"
                          r"betel|shupari|paan|chal kumra|kochu|arum|shim|motorshuti|lemongrass|drumstick|broccoli|\bpui\b|microgreen|sheem"),
]
NONFOOD_PREFIX = "NF: "
_COMPILED = [(c, re.compile(p, re.I)) for c, p in CATEGORY_RULES]

def fix_mojibake(s: str) -> str:
    """Repair UTF-8 read as cp1252 (e.g. 'Â±50 gm' -> '±50 gm')."""
    try:
        t = s.encode("cp1252").decode("utf-8")
        return t
    except Exception:
        return s.replace("Â±", "±").replace("Â ", " ").replace("â€™", "'")

def tag_category(name: str) -> str:
    n = " " + name.lower().strip() + " "
    for cat, rx in _COMPILED:
        if rx.search(n):
            return cat
    return "Unclassified"

# ---------------------------------------------------------------------------
# 1.2  Load the wide panel  (Name | date1 | date2 | ...)
# ---------------------------------------------------------------------------
def load_panel(path):
    path = Path(path)
    if path.suffix.lower() in (".xlsx", ".xls"):
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path, encoding="utf-8-sig")
    df = df.rename(columns={df.columns[0]: "Name"})
    date_cols = [c for c in df.columns[1:]]
    dates = pd.to_datetime([str(c)[:10] for c in date_cols])
    P = df[date_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float64)
    names = df["Name"].astype(str).map(fix_mojibake).str.strip().tolist()
    return names, dates, P

# ---------------------------------------------------------------------------
# 1.3  Data-quality: interpolated-window mask, spike-and-revert repair,
#      persistent variant breaks, platform batch-update days
# ---------------------------------------------------------------------------
def detect_carried_forward_days(P, max_changes=2):
    """Scraper-outage days. When the scraper did not run, the panel carries the
    previous day's prices forward, so NOT A SINGLE product changes price that
    day (the median day has ~80 changes). These cells are imputed, not observed:
    they are excluded from every loss/metric target and from graph statistics.
    The first observed day after an outage accumulates several days of changes
    (a 'gap-end' day) and is flagged separately."""
    ch = np.concatenate([[10**9], (np.diff(P, axis=1) != 0).sum(axis=0)])
    cf = ch <= max_changes
    gap_end = np.zeros_like(cf)
    gap_end[1:] = cf[:-1] & ~cf[1:]
    return cf, gap_end

def repair_spikes(P, tau=np.log(1.5), max_len=7, tol=0.05):
    """Spike-and-revert repair: a jump |dlog| > tau that returns to within `tol`
    of the pre-jump level within `max_len` days is treated as a scrape error
    (wrong variant / pack size shown for a few days) and replaced by the
    pre-jump level. Returns cleaned matrix + per-series repair counts."""
    L = np.log(P)
    Q = P.copy(); n_rep = np.zeros(P.shape[0], dtype=int)
    T = P.shape[1]
    for i in range(P.shape[0]):
        x = L[i]
        d = np.diff(x)
        cand = np.where(np.abs(d) > tau)[0]
        t_skip = -1
        for j in cand:
            if j <= t_skip: continue
            base = x[j]
            for k in range(j + 1, min(j + 1 + max_len, T - 1)):
                if abs(x[k + 1] - base) <= tol:
                    Q[i, j + 1: k + 1] = P[i, j]
                    n_rep[i] += 1; t_skip = k
                    break
    return Q, n_rep

def detect_variant_breaks(P, t_end, thr=np.log(3.0)):
    """Persistent level shifts > x3 inside the training window: almost surely a
    re-used product slot (different pack/variant), not a price change."""
    d = np.abs(np.diff(np.log(P[:, : t_end + 1]), axis=1))
    return (d > thr).any(axis=1)

def detect_batch_days(P, t_end, k_mad=5.0):
    """Days where the number of simultaneous price changes is an outlier
    (median + k*MAD, threshold estimated on training days only)."""
    ch = (np.diff(P, axis=1) != 0).sum(axis=0)          # changes on day t (t>=1)
    ch = np.concatenate([[0], ch])
    tr = ch[1: t_end + 1]
    med = np.median(tr); mad = np.median(np.abs(tr - med)) + 1e-9
    thr = med + k_mad * 1.4826 * mad
    return ch > thr, ch, thr

# ---------------------------------------------------------------------------
# 1.4  Rolling-origin splits (3 folds, 30-day val + 30-day test each)
# ---------------------------------------------------------------------------
def make_folds(T, n_folds=3, test_len=30, val_len=30):
    folds = []
    for k in range(n_folds):
        test_end = T - 1 - test_len * (n_folds - 1 - k)
        test_start = test_end - test_len + 1
        val_end = test_start - 1; val_start = val_end - val_len + 1
        folds.append(dict(fold=k, train_end=val_start - 1, val_start=val_start, val_end=val_end,
                          test_start=test_start, test_end=test_end))
    return folds

# ---------------------------------------------------------------------------
# 1.5  Master prep (cached to Drive; identical in all five notebooks)
# ---------------------------------------------------------------------------
PREP_VERSION = "v1.1"

def build_prep(data_path, cache_dir, clean_spikes=True, force=False):
    cache_dir = Path(cache_dir); cache_dir.mkdir(parents=True, exist_ok=True)
    f = cache_dir / f"prep_{PREP_VERSION}.pkl"
    if f.exists() and not force:
        with open(f, "rb") as fh:
            prep = pickle.load(fh)
        print(f"[prep] loaded cached panel from {f}")
        return prep
    t0 = time.time()
    names, dates, P_raw = load_panel(data_path)
    N, T = P_raw.shape
    assert np.isfinite(P_raw).all() and (P_raw > 0).all(), "panel must be complete & positive"
    folds = make_folds(T)
    tier_end = folds[0]["train_end"]                    # leak-free: earliest training window
    cf_day, gap_end = detect_carried_forward_days(P_raw)
    P_clean, n_rep = repair_spikes(P_raw) if clean_spikes else (P_raw.copy(), np.zeros(N, int))
    variant_break = detect_variant_breaks(P_clean, tier_end)
    batch_day, daily_changes, batch_thr = detect_batch_days(P_clean, tier_end)
    cats = [tag_category(n) for n in names]
    is_food = np.array([not c.startswith(NONFOOD_PREFIX) and c != "Unclassified" for c in cats])
    # activity statistics on the tier window only (no look-ahead)
    dlog_tr = np.diff(np.log(P_clean[:, : tier_end + 1]), axis=1)
    n_changes = (dlog_tr != 0).sum(axis=1)
    vol = dlog_tr.std(axis=1)
    usable = is_food & ~variant_break
    tier = np.full(N, "T0", dtype=object)
    tier[usable & (n_changes >= 1)] = "T1"
    tier[usable & (n_changes >= 10)] = "T2"
    # volatility terciles among T1 (training-window std of log-returns)
    vol_bucket = np.full(N, "none", dtype=object)
    m = usable & (n_changes >= 1)
    q1, q2 = np.quantile(vol[m], [1 / 3, 2 / 3])
    vol_bucket[m & (vol <= q1)] = "low"
    vol_bucket[m & (vol > q1) & (vol <= q2)] = "medium"
    vol_bucket[m & (vol > q2)] = "high"
    meta = pd.DataFrame(dict(name=names, category=cats, is_food=is_food, variant_break=variant_break,
                             n_changes_train=n_changes, vol_train=vol, tier=tier, vol_bucket=vol_bucket,
                             spike_repairs=n_rep, mean_price_train=P_clean[:, : tier_end + 1].mean(axis=1)))
    prep = dict(names=names, dates=dates, P_raw=P_raw, P=P_clean, logP=np.log(P_clean), meta=meta,
                folds=folds, tier_end=tier_end, cf_day=cf_day, gap_end=gap_end,
                batch_day=batch_day, daily_changes=daily_changes, batch_thr=batch_thr,
                version=PREP_VERSION, built=time.strftime("%Y-%m-%d %H:%M"))
    with open(f, "wb") as fh:
        pickle.dump(prep, fh)
    print(f"[prep] built in {time.time()-t0:.1f}s and cached to {f}")
    return prep

def subset_idx(prep, tiers=("T1", "T2"), food_only=True, smoke_n=None, seed=0):
    meta = prep["meta"]
    m = meta.tier.isin(tiers).to_numpy()
    if food_only: m &= meta.is_food.to_numpy()
    idx = np.where(m)[0]
    if smoke_n is not None and len(idx) > smoke_n:
        rng = np.random.default_rng(seed); idx = np.sort(rng.choice(idx, smoke_n, replace=False))
    return idx

In [ ]:
# ============================================================================
# SECTION 1b — SHARED UTILITIES: figures, tables, checkpoints, windows, metrics
# ============================================================================
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats as sps

# ---------------------------------------------------------------------------
# Figure style (journal): 8-pt sans, thin marks, recessive grid, fixed palette
# ---------------------------------------------------------------------------
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
GREY = "#8a8985"
COL1, COL2 = 3.5, 7.2                       # single / double column widths (inches)
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
    "font.size": 8, "axes.titlesize": 8.5, "axes.labelsize": 8, "legend.fontsize": 7,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "axes.spines.top": False, "axes.spines.right": False,
    "axes.linewidth": 0.6, "axes.edgecolor": "#52514e", "axes.grid": True, "grid.color": "#e4e3df",
    "grid.linewidth": 0.5, "lines.linewidth": 1.4, "lines.markersize": 4, "legend.frameon": False,
    "axes.prop_cycle": matplotlib.cycler(color=PALETTE), "pdf.fonttype": 42, "ps.fonttype": 42,
})

def savefig(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIG_DIR / f"{name}.{ext}")
    print(f"[fig] {name} -> {FIG_DIR}")
    plt.show(); plt.close(fig)

# ---------------------------------------------------------------------------
# Tables: CSV + LaTeX (booktabs), no jinja2 dependency; optional bolding of best
# ---------------------------------------------------------------------------
def _fmt(v, nd):
    if isinstance(v, (float, np.floating)):
        if not np.isfinite(v): return "--"
        return f"{v:.{nd}f}"
    return str(v).replace("_", r"\_").replace("&", r"\&").replace("%", r"\%")

def save_table(df, name, caption="", label=None, nd=3, bold_min=None, bold_max=None, index=False, bold_row_min=None):
    """Write <name>.csv and <name>.tex (booktabs). bold_min/bold_max: list of
    columns in which the best value is set in bold."""
    df = df.reset_index() if index else df.copy()
    df.to_csv(TAB_DIR / f"{name}.csv", index=False)
    best = {}
    for c in (bold_min or []):
        if c in df: best[c] = pd.to_numeric(df[c], errors="coerce").min()
    for c in (bold_max or []):
        if c in df: best[c] = pd.to_numeric(df[c], errors="coerce").max()
    cols = list(df.columns)
    align = "".join("l" if not np.issubdtype(df[c].dtype, np.number) else "r" for c in cols)
    lines = [r"\begin{table}[t]", r"\centering", r"\small"]
    if caption: lines.append(rf"\caption{{{caption}}}")
    lines.append(rf"\label{{{label or 'tab:' + name}}}")
    lines += [rf"\begin{{tabular}}{{{align}}}", r"\toprule",
              " & ".join(_fmt(c, nd) for c in cols) + r" \\", r"\midrule"]
    for _, r in df.iterrows():
        cells = []
        rmin = pd.to_numeric(r[bold_row_min], errors="coerce").min() if bold_row_min else None
        for c in cols:
            s = _fmt(r[c], nd)
            if np.issubdtype(df[c].dtype, np.integer): s = str(int(r[c]))
            if bold_row_min and c in bold_row_min and np.isfinite(r[c]) and np.isclose(r[c], rmin):
                s = rf"\textbf{{{s}}}"
            elif c in best and isinstance(r[c], (float, np.floating, int, np.integer)) and np.isfinite(r[c]) \
               and np.isclose(r[c], best[c]):
                s = rf"\textbf{{{s}}}"
            cells.append(s)
        lines.append(" & ".join(cells) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    (TAB_DIR / f"{name}.tex").write_text("\n".join(lines))
    print(f"[table] {name} -> {TAB_DIR}")
    try:
        display(df)
    except NameError:
        print(df.to_string())
    return df

# ---------------------------------------------------------------------------
# Result caching & checkpoints (everything resumable after a Colab disconnect)
# ---------------------------------------------------------------------------
def cached(name, fn, force=False):
    f = RES_DIR / f"{name}.pkl"
    if f.exists() and not force:
        with open(f, "rb") as fh: return pickle.load(fh)
    out = fn()
    with open(f, "wb") as fh: pickle.dump(out, fh)
    return out

def set_seed(s):
    import random
    random.seed(s); np.random.seed(s)
    try:
        import torch
        torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except ImportError:
        pass

# ---------------------------------------------------------------------------
# Feature tensor & rolling windows
#   channels: 0 = 100*log(P_t'/P_origin)   (filled at gather time)
#             1 = price-changed indicator  2 = observed (not carried-forward)
#             3,4 = day-of-week sin/cos     5 = log1p(days since last change)/5
#   targets : 100*log(P_{o+h}/P_o), h = 1..H, clipped to [-100, 100]
# ---------------------------------------------------------------------------
H_MAX, H_EVAL = 30, [1, 3, 7, 14, 30]
N_CH = 6

def build_features(prep, idx):
    lp = prep["logP"][idx]
    n, T = lp.shape
    chg = np.zeros_like(lp); chg[:, 1:] = (np.diff(lp, axis=1) != 0)
    obs = np.broadcast_to((~prep["cf_day"]).astype(np.float64), (n, T))
    dow = prep["dates"].dayofweek.to_numpy()
    s, c = np.sin(2 * np.pi * dow / 7), np.cos(2 * np.pi * dow / 7)
    dsc = np.zeros_like(lp)
    for t in range(1, T):
        dsc[:, t] = np.where(chg[:, t] > 0, 0, dsc[:, t - 1] + 1)
    F = np.stack([lp, chg, obs, np.broadcast_to(s, (n, T)), np.broadcast_to(c, (n, T)),
                  np.log1p(dsc) / 5], axis=-1).astype(np.float32)
    return F                                                   # (n, T, C); channel 0 = raw log-price

def origins_for(fold, part, L, H=H_MAX):
    """Forecast origins o (inclusive) for 'train' | 'val' | 'test'. Targets at
    o+1..o+H are masked beyond the segment end (no peeking)."""
    if part == "train":
        return np.arange(L - 1, fold["train_end"]), fold["train_end"]
    if part == "val":
        return np.arange(fold["val_start"] - 1, fold["val_end"]), fold["val_end"]
    return np.arange(fold["test_start"] - 1, fold["test_end"]), fold["test_end"]

def gather_windows(F, idx, origins, L, H, seg_end, cf_day, rows=None):
    """Vectorised, ORIGIN-MAJOR gather for products `rows` (local indices into F,
    default all) at forecast origins `origins`.
    Returns X (k, n, L, C), Y (k, n, H), M (k, n, H) and meta (k, n, 2) holding
    (global product id, origin day). Channel 0 of X is re-based to the origin."""
    rows = np.arange(F.shape[0]) if rows is None else np.asarray(rows)
    oo, rr = np.meshgrid(origins, rows, indexing="ij")                    # (k, n)
    tt = oo[..., None] + np.arange(-L + 1, 1)                              # (k, n, L)
    X = F[rr[..., None], np.maximum(tt, 0)].copy()
    base = F[rr, oo, 0]
    X[..., 0] = 100.0 * (X[..., 0] - base[..., None])
    hh = oo[..., None] + np.arange(1, H + 1)                               # (k, n, H)
    valid = hh <= seg_end
    hc = np.minimum(hh, F.shape[1] - 1)
    M = valid & ~cf_day[hc]
    Y = 100.0 * (F[rr[..., None], hc, 0] - base[..., None])
    Y = np.clip(np.where(M, Y, 0.0), -100, 100).astype(np.float32)
    meta = np.stack([np.asarray(idx)[rr], oo], -1)
    return X.astype(np.float32), Y, M, meta

# ---------------------------------------------------------------------------
# Evaluation in PRICE space. Predictions are 100*log-ratio vs the origin price.
# ---------------------------------------------------------------------------
def price_scale(prep, idx, fold):
    """MASE scale: mean |P_t - P_{t-1}| on the fold's training window."""
    P = prep["P"][idx, : fold["train_end"] + 1]
    return np.maximum(np.abs(np.diff(P, axis=1)).mean(axis=1), 1e-6)

def evaluate(pred, Y, M, meta, prep, fold, horizons=H_EVAL):
    """pred, Y, M: (..., H) arrays (any leading shape); meta (..., 2).
    Returns (summary per horizon, {h: per-product mean sAPE})."""
    H = Y.shape[-1]
    pred, Y, M, meta = pred.reshape(-1, H), Y.reshape(-1, H), M.reshape(-1, H), meta.reshape(-1, 2)
    P0 = prep["P"][meta[:, 0], meta[:, 1]]
    gids = np.unique(meta[:, 0])
    scale = pd.Series(price_scale(prep, gids, fold), index=gids)
    sc = scale.loc[meta[:, 0]].to_numpy()
    rows = []
    per_prod = {}
    for h in horizons:
        m = M[:, h - 1]
        if m.sum() == 0: continue
        y = P0[m] * np.exp(Y[m, h - 1] / 100); yh = P0[m] * np.exp(pred[m, h - 1] / 100)
        e = yh - y; ae = np.abs(e)
        sape = 200 * ae / (np.abs(y) + np.abs(yh))
        chg = np.abs(Y[m, h - 1]) > 1e-9
        prod = meta[m, 0]
        df = pd.DataFrame(dict(prod=prod, ae=ae, sape=sape, se=ae**2, ape=100 * ae / y, sc_ae=ae / sc[m], y=y))
        g = df.groupby("prod")
        mase_p = g.sc_ae.mean()
        rows.append(dict(h=h, n=int(m.sum()), MAE=ae.mean(), RMSE=np.sqrt((ae**2).mean()),
                         MAPE=df.ape.mean(), sMAPE=sape.mean(), WAPE=100 * ae.sum() / y.sum(),
                         MASE=mase_p.mean(), MASE_med=mase_p.median(),
                         CC_sMAPE=sape[chg].mean() if chg.any() else np.nan,
                         change_rate=chg.mean()))
        per_prod[h] = g.sape.mean()
    return pd.DataFrame(rows), per_prod

def change_auprc(score, Y, M, h):
    """AUPRC for 'will the price differ from today's price at horizon h?'."""
    from sklearn.metrics import average_precision_score
    H = Y.shape[-1]; score, Y, M = score.reshape(-1, H), Y.reshape(-1, H), M.reshape(-1, H)
    m = M[:, h - 1]; lab = np.abs(Y[m, h - 1]) > 1e-9
    if lab.sum() == 0 or lab.all(): return np.nan
    return average_precision_score(lab, score[m, h - 1])

# ---------------------------------------------------------------------------
# Statistical tests
# ---------------------------------------------------------------------------
def dm_test(l1, l2, h=1):
    """Diebold-Mariano with Harvey-Leybourne-Newbold small-sample correction.
    l1, l2: loss series (e.g. cross-sectional mean sAPE per origin). Negative
    statistic => model 1 more accurate."""
    d = np.asarray(l1) - np.asarray(l2); d = d[np.isfinite(d)]
    T = len(d)
    if T < 5: return np.nan, np.nan
    dbar = d.mean(); g = [np.mean((d[k:] - dbar) * (d[: T - k] - dbar)) for k in range(min(h, T - 1))]
    v = (g[0] + 2 * sum(g[1:])) / T
    if v <= 0: return np.nan, np.nan
    stat = dbar / np.sqrt(v) * np.sqrt((T + 1 - 2 * h + h * (h - 1) / T) / T)
    return stat, 2 * sps.t.sf(abs(stat), df=T - 1)

def wilcoxon_paired(a, b):
    a, b = pd.Series(a), pd.Series(b); j = a.index.intersection(b.index)
    d = (a[j] - b[j]).to_numpy()
    if np.allclose(d, 0): return np.nan
    return sps.wilcoxon(d, zero_method="zsplit").pvalue

def bootstrap_ci(d, B=2000, seed=0, alpha=0.05):
    d = np.asarray(d); d = d[np.isfinite(d)]
    rng = np.random.default_rng(seed)
    bs = rng.choice(d, (B, len(d)), replace=True).mean(axis=1)
    return d.mean(), np.quantile(bs, alpha / 2), np.quantile(bs, 1 - alpha / 2)

def holm(pvals):
    p = np.asarray(pvals, dtype=float); out = np.full_like(p, np.nan)
    ok = np.isfinite(p); idx = np.argsort(p[ok]); m = ok.sum()
    adj = np.maximum.accumulate((m - np.arange(m)) * p[ok][idx]); adj = np.minimum(adj, 1)
    tmp = np.empty(m); tmp[idx] = adj; out[ok] = tmp
    return out

def per_origin_loss(pred, Y, M, meta, prep, h):
    """Cross-sectional mean sAPE per origin at horizon h (DM-test input)."""
    H = Y.shape[-1]
    pred, Y, M, meta = pred.reshape(-1, H), Y.reshape(-1, H), M.reshape(-1, H), meta.reshape(-1, 2)
    m = M[:, h - 1]
    P0 = prep["P"][meta[m, 0], meta[m, 1]]
    y = P0 * np.exp(Y[m, h - 1] / 100); yh = P0 * np.exp(pred[m, h - 1] / 100)
    s = pd.Series(200 * np.abs(yh - y) / (y + yh)).groupby(meta[m, 1]).mean()
    return s

# ---------------------------------------------------------------------------
# Dataset description (figure + table used in the Data section of every paper)
# ---------------------------------------------------------------------------
def dataset_overview(prep):
    meta, dates = prep["meta"], prep["dates"]
    rows = [("Products (SKUs)", len(meta)), ("Days", len(dates)),
            ("Period", f"{dates[0].date()} to {dates[-1].date()}"),
            ("Food SKUs (tagged)", int(meta.is_food.sum())),
            ("Excluded: variant breaks", int(meta.variant_break.sum())),
            ("Scraper-outage days (carried forward)", int(prep["cf_day"].sum())),
            ("Platform batch-update days", int(prep["batch_day"].sum())),
            ("Spike-and-revert repairs", int(meta.spike_repairs.sum())),
            ("T0 (food, never changes in train)", int(((meta.tier == "T0") & meta.is_food).sum())),
            ("T1 (food, 1-9 changes)", int((meta.tier == "T1").sum())),
            ("T2 (food, >=10 changes)", int((meta.tier == "T2").sum()))]
    save_table(pd.DataFrame(rows, columns=["Item", "Value"]).astype({"Value": str}), "data_summary",
               "Dataset summary after quality control.")
    fig, ax = plt.subplots(1, 3, figsize=(COL2, 2.2))
    ch = prep["daily_changes"]
    ax[0].bar(dates, ch, width=1.0, color=PALETTE[0], lw=0)
    ax[0].bar(dates[prep["batch_day"]], ch[prep["batch_day"]], width=2.5, color=PALETTE[1], lw=0, label="batch update")
    ax[0].scatter(dates[prep["cf_day"]], np.zeros(prep["cf_day"].sum()), marker="|", s=30, color=GREY, label="outage (0 changes)")
    ax[0].set_title("(a) Daily number of price changes"); ax[0].legend(loc="upper left")
    ax[0].tick_params(axis="x", rotation=30)
    nc = meta.loc[meta.is_food, "n_changes_train"]
    ax[1].hist(np.log10(nc + 1), bins=30, color=PALETTE[0], edgecolor="white", lw=0.4)
    ax[1].set_xlabel("log$_{10}$(1 + price changes in training window)"); ax[1].set_ylabel("Food SKUs")
    ax[1].set_title("(b) Price stickiness")
    act = meta[meta.tier.isin(["T1", "T2"])]
    order = act.category.value_counts()
    ax[2].barh(order.index[::-1], order.values[::-1], color=PALETTE[0])
    ax[2].set_title("(c) Active food SKUs by category"); ax[2].tick_params(axis="y", labelsize=6)
    fig.tight_layout(); savefig(fig, "fig_data_overview")

# ---------------------------------------------------------------------------
# Fold data bundle: windows for train / val / test in one call (cached in RAM)
# ---------------------------------------------------------------------------
def fold_data(prep, F, idx, fold, L, H=H_MAX, parts=("train", "val", "test"), train_stride=1):
    out = {}
    for part in parts:
        O, seg_end = origins_for(fold, part, L, H)
        if part == "train": O = O[::train_stride]
        out[part] = gather_windows(F, idx, O, L, H, seg_end, prep["cf_day"])
    return out

def naive_family(X, H=H_MAX):
    """Naive (0), seasonal-naive(7) and moving-average(7) forecasts from windows.
    X channel 0 is 100*log(P/P_origin) so the naive forecast is identically 0."""
    k = X[..., 0]
    naive = np.zeros(k.shape[:-1] + (H,), np.float32)
    L = k.shape[-1]
    snaive = np.stack([k[..., L - 7 + ((h - 1) % 7)] for h in range(1, H + 1)], -1)
    ma = np.repeat(k[..., -7:].mean(-1, keepdims=True), H, -1)
    return {"Naive": naive, "SNaive-7": snaive.astype(np.float32), "MA-7": ma.astype(np.float32)}

def results_frame(res_dict):
    """{model: summary-df} -> long frame with a 'model' column."""
    return pd.concat([d.assign(model=m) for m, d in res_dict.items()], ignore_index=True)

In [ ]:
# ============================================================================
# SECTION 1c — SHARED PYTORCH UTILITIES: resumable trainer + building blocks
# ============================================================================
import torch, torch.nn as nn, torch.nn.functional as Fnn
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
print("device:", DEVICE)

def masked_huber(pred, y, m, delta=1.0):
    m = m.float()
    l = Fnn.huber_loss(pred, y, reduction="none", delta=delta)
    return (l * m).sum() / m.sum().clamp(min=1.0)

def iterate_flat(X, Y, M, bs, shuffle, rng=None, extra=None):
    """Mini-batches over flattened (origin, product) windows. X: (k,n,L,C).
    extra: dict of (k,n,...) arrays gathered alongside (e.g. category ids)."""
    k, n = X.shape[:2]
    N = k * n
    order = (rng or np.random.default_rng(0)).permutation(N) if shuffle else np.arange(N)
    Xf, Yf, Mf = X.reshape(N, *X.shape[2:]), Y.reshape(N, -1), M.reshape(N, -1)
    ex = {kk: v.reshape(N, *v.shape[2:]) for kk, v in (extra or {}).items()}
    for s in range(0, N, bs):
        j = order[s: s + bs]
        b = dict(x=torch.from_numpy(Xf[j]).to(DEVICE), y=torch.from_numpy(Yf[j]).to(DEVICE),
                 m=torch.from_numpy(Mf[j]).to(DEVICE))
        for kk, v in ex.items(): b[kk] = torch.from_numpy(v[j]).to(DEVICE)
        yield b

def iterate_snap(X, Y, M, bs, shuffle, rng=None, extra=None):
    """Mini-batches of whole cross-sections (all products at an origin) for
    graph / cross-product models. X: (k, n, L, C) -> batch x: (b, n, L, C)."""
    k = X.shape[0]
    order = (rng or np.random.default_rng(0)).permutation(k) if shuffle else np.arange(k)
    for s in range(0, k, bs):
        j = order[s: s + bs]
        b = dict(x=torch.from_numpy(X[j]).to(DEVICE), y=torch.from_numpy(Y[j]).to(DEVICE),
                 m=torch.from_numpy(M[j]).to(DEVICE))
        for kk, v in (extra or {}).items(): b[kk] = torch.from_numpy(v[j]).to(DEVICE)
        yield b

def fit(model, train_iter, val_iter, loss_fn, ckpt_path, epochs=30, lr=1e-3, wd=1e-4,
        patience=6, verbose=True, clip=1.0):
    """Resumable training loop. `train_iter(epoch)` / `val_iter()` yield batches,
    `loss_fn(model, batch)` returns a scalar loss. A checkpoint is written after
    every epoch (model, optimiser, scheduler, history, early-stopping state);
    re-running the cell resumes where it stopped; a finished run is skipped."""
    ckpt_path = Path(ckpt_path); ckpt_path.parent.mkdir(parents=True, exist_ok=True)
    model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    st = dict(epoch=0, best=np.inf, bad=0, hist=[], done=False, best_state=None)
    if ckpt_path.exists():
        ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        st = ck["state"]
        if st["done"]:
            model.load_state_dict(st["best_state"]); model.eval()
            return model, pd.DataFrame(st["hist"])
        if verbose: print(f"  resuming {ckpt_path.name} at epoch {st['epoch']}")
    for ep in range(st["epoch"], epochs):
        model.train(); t0 = time.time(); tl, nb = 0.0, 0
        for b in train_iter(ep):
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, enabled=AMP, dtype=torch.float16):
                loss = loss_fn(model, b)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            scaler.step(opt); scaler.update(); tl += loss.item(); nb += 1
        sched.step()
        model.eval(); vl, vb = 0.0, 0
        with torch.no_grad():
            for b in val_iter():
                with torch.autocast(DEVICE.type, enabled=AMP, dtype=torch.float16):
                    vl += loss_fn(model, b).item(); vb += 1
        vl /= max(vb, 1); tl /= max(nb, 1)
        st["hist"].append(dict(epoch=ep, train=tl, val=vl, sec=time.time() - t0))
        if vl < st["best"] - 1e-5:
            st["best"], st["bad"] = vl, 0
            st["best_state"] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            st["bad"] += 1
        st["epoch"] = ep + 1
        stop = st["bad"] >= patience or st["epoch"] >= epochs
        st["done"] = stop
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), state=st), ckpt_path)
        if verbose: print(f"  ep {ep:02d} train {tl:.4f} val {vl:.4f} ({time.time()-t0:.1f}s)")
        if stop: break
    if st["best_state"] is not None: model.load_state_dict(st["best_state"])
    model.eval()
    return model, pd.DataFrame(st["hist"])

@torch.no_grad()
def predict(model, batches, key="x", fwd=None):
    model.eval(); out = []
    for b in batches:
        with torch.autocast(DEVICE.type, enabled=AMP, dtype=torch.float16):
            p = fwd(model, b) if fwd else model(b[key])
        out.append(p.float().cpu().numpy())
    return np.concatenate(out, 0)

# ---------------------------------------------------------------------------
# Building blocks shared by several papers
# ---------------------------------------------------------------------------
class PatchEncoder(nn.Module):
    """PatchTST-style temporal encoder: (B, L, C) -> (B, d)."""
    def __init__(self, L, C, d=64, patch=6, layers=2, heads=4, dropout=0.1):
        super().__init__()
        assert L % patch == 0
        self.patch, self.np = patch, L // patch
        self.proj = nn.Linear(patch * C, d)
        self.pos = nn.Parameter(torch.randn(1, self.np, d) * 0.02)
        el = nn.TransformerEncoderLayer(d, heads, 2 * d, dropout, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(el, layers, enable_nested_tensor=False)
        self.out = nn.Sequential(nn.LayerNorm(2 * d), nn.Linear(2 * d, d))
    def tokens(self, x):
        B, L, C = x.shape
        z = x.reshape(B, self.np, self.patch * C)
        return self.enc(self.proj(z) + self.pos)                 # (B, np, d)
    def forward(self, x):
        h = self.tokens(x)
        return self.out(torch.cat([h.mean(1), h[:, -1]], -1))

class GRUEncoder(nn.Module):
    def __init__(self, C, d=64, layers=1):
        super().__init__(); self.rnn = nn.GRU(C, d, layers, batch_first=True)
    def forward(self, x): return self.rnn(x)[0][:, -1]

class LSTMEncoder(nn.Module):
    def __init__(self, C, d=64, layers=2, dropout=0.1):
        super().__init__(); self.rnn = nn.LSTM(C, d, layers, batch_first=True, dropout=dropout)
    def forward(self, x): return self.rnn(x)[0][:, -1]

class MLPEncoder(nn.Module):
    def __init__(self, L, C, d=64):
        super().__init__(); self.net = nn.Sequential(nn.Flatten(), nn.Linear(L * C, 2 * d), nn.GELU(), nn.Linear(2 * d, d))
    def forward(self, x): return self.net(x)

class Head(nn.Module):
    def __init__(self, d, H, dropout=0.1):
        super().__init__(); self.net = nn.Sequential(nn.LayerNorm(d), nn.Dropout(dropout), nn.Linear(d, d), nn.GELU(), nn.Linear(d, H))
    def forward(self, h): return self.net(h)

In [ ]:
# ==== 1d. BUILD / LOAD THE CLEAN PANEL (cached on Drive) ====================
prep = build_prep(DATA_PATH, PREP_DIR)
meta, FOLDS = prep["meta"], prep["folds"]
CATS = sorted(meta.category.unique()); CAT2ID = {c: i for i, c in enumerate(CATS)}
for f in FOLDS:
    d = prep["dates"]
    print(f"fold {f['fold']}: train ..{d[f['train_end']].date()} | val {d[f['val_start']].date()}..{d[f['val_end']].date()}"
          f" | test {d[f['test_start']].date()}..{d[f['test_end']].date()}")
print(meta.tier.value_counts().to_dict())

In [ ]:
# ==== 2. DATASET OVERVIEW ===================================================
dataset_overview(prep)

In [ ]:
# ==== 3. CONFIGURATION + SHOCK LABELS =======================================
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve, brier_score_loss
L, H = 60, H_MAX
HS = [1, 7, 14]
IDX = subset_idx(prep, ("T1", "T2"), smoke_n=80 if SMOKE else None)
N = len(IDX)
CAT = np.array([CAT2ID[c] for c in meta.category.to_numpy()[IDX]])
F = build_features(prep, IDX)
FOLD_IDS = [1, 2] if SMOKE else [0, 1, 2]
SEEDS = [0] if SMOKE else [0, 1, 2]
EPOCHS = 2 if SMOKE else 40
D = {f: fold_data(prep, F, IDX, FOLDS[f], L) for f in FOLD_IDS}
PRED_DIR = RES_DIR / "preds"; PRED_DIR.mkdir(exist_ok=True)

def thresholds(f):
    """theta(c, h) from training-window positive h-day log changes of each category."""
    te = FOLDS[f]["train_end"]; lp = prep["logP"][IDX]; th = np.full((len(CATS), len(HS)), 5.0)
    for j, h in enumerate(HS):
        r = 100 * (lp[:, h: te + 1] - lp[:, : te + 1 - h])
        for c in np.unique(CAT):
            pos = r[CAT == c]; pos = pos[pos > 0]
            if len(pos) >= 10: th[c, j] = max(5.0, float(np.median(pos)))
    return th

TH = {f: thresholds(f) for f in FOLD_IDS}
def labels(Y, M, f):
    """Y, M: (k, n, H) -> shock labels S (k, n, 3), mask SM, volatility target V (k, n)."""
    th = TH[f][CAT]                                                        # (n, 3)
    S = np.stack([(Y[..., h - 1] >= th[None, :, j]) for j, h in enumerate(HS)], -1).astype(np.float32)
    SM = np.stack([M[..., h - 1] for h in HS], -1)
    m14 = M[..., :14]
    V = np.log1p((np.abs(Y[..., :14]) * m14).sum(-1) / np.maximum(m14.sum(-1), 1)).astype(np.float32)
    return S, SM, V

LAB = {f: {p: labels(D[f][p][1], D[f][p][2], f) for p in ("train", "val", "test")} for f in FOLD_IDS}
rows = []
for f in FOLD_IDS:
    for p in ("train", "val", "test"):
        S, SM, _ = LAB[f][p]
        rows.append(dict(fold=f, part=p, **{f"rate_h{h}": S[..., j][SM[..., j]].mean() for j, h in enumerate(HS)},
                         **{f"n_pos_h{h}": int(S[..., j][SM[..., j]].sum()) for j, h in enumerate(HS)}))
save_table(pd.DataFrame(rows), "tab_shock_prevalence", "Shock prevalence (share of valid product-origin pairs).", nd=4)
th_tab = pd.DataFrame(TH[FOLD_IDS[-1]], index=CATS, columns=[f"theta_h{h}" for h in HS]).loc[sorted(set(np.array(CATS)[CAT]))]
save_table(th_tab.reset_index().rename(columns={"index": "category"}), "tab_shock_thresholds",
           "Category-specific shock thresholds theta(c, h) (percent log-change), latest fold.", nd=2)

## 4. Features for the tabular baselines
Own-history features (lags, rolling moments, change counts, days since last change) plus **cross-product
signals at the origin**: the category's mean return and share of increasing products over the last 1/3/7
days, platform-wide number of price changes and share of increases over the last 1/3/7 days, and whether a
batch-update or outage day occurred in the last 7 days.

In [ ]:
# ==== 4. FEATURES ===========================================================
def own_features(X):
    v = X[..., 0]; c = X[..., 1]
    feats = [v[..., -14:-1][..., ::-1]]
    for w in (7, 14, 30, 60):
        feats += [v[..., -w:].mean(-1, keepdims=True), v[..., -w:].std(-1, keepdims=True),
                  (v[..., -w:].max(-1) - v[..., -w:].min(-1))[..., None]]
    feats += [c[..., -7:].sum(-1, keepdims=True), c[..., -30:].sum(-1, keepdims=True), c.sum(-1, keepdims=True), X[..., -1, 5:6]]
    return np.concatenate(feats, -1)

def cross_features(meta_arr):
    """meta_arr (k, n, 2) -> cross-product features (k, n, F) computed at each origin."""
    O = meta_arr[:, 0, 1]; lp = prep["logP"][IDX]; out = []
    for o in O:
        per = []
        for w in (1, 3, 7):
            r = 100 * (lp[:, o] - lp[:, max(o - w, 0)])
            cm = pd.Series(r).groupby(CAT).transform("mean").to_numpy()
            cu = pd.Series(r > 0).groupby(CAT).transform("mean").to_numpy()
            per += [cm, cu, np.full(N, (r != 0).mean()), np.full(N, (r > 0).mean())]
        per += [np.full(N, prep["batch_day"][max(o - 6, 0): o + 1].any()), np.full(N, prep["cf_day"][max(o - 6, 0): o + 1].mean()),
                np.full(N, np.sin(2 * np.pi * prep["dates"][o].dayofweek / 7))]
        out.append(np.stack(per, -1))
    return np.stack(out).astype(np.float32)

def tab(f, p, cross=True):
    X, _, _, mt = D[f][p]
    Z = [own_features(X), CAT[None, :, None].repeat(X.shape[0], 0).astype(np.float32)]
    if cross: Z.append(cross_features(mt))
    Z = np.concatenate(Z, -1); return Z.reshape(-1, Z.shape[-1])

## 5. Models

In [ ]:
# ==== 5. BASELINES: persistence, logistic regression, LightGBM (own vs own+cross features) ====
SCORES = {f: {} for f in FOLD_IDS}           # SCORES[f][model] = list over seeds of (val_scores, test_scores), each (k, n, 3)

def persistence(f, p):
    X = D[f][p][0]; v = X[..., 0]
    s7 = np.maximum(-v[..., -8], 0)             # own increase over the last 7 days (in origin-relative units)
    last = X[..., -1, 1]                        # changed today
    return np.stack([s7 + last, s7 + last, s7 + last], -1)

def tabular_model(f, kind, cross, seed=0):
    def run():
        Ztr, Zva, Zte = tab(f, "train", cross), tab(f, "val", cross), tab(f, "test", cross)
        outs = [np.zeros(LAB[f][p][0].shape, np.float32) for p in ("val", "test")]
        for j in range(len(HS)):
            y = LAB[f]["train"][0][..., j].ravel(); m = LAB[f]["train"][1][..., j].ravel()
            yv = LAB[f]["val"][0][..., j].ravel(); mv = LAB[f]["val"][1][..., j].ravel()
            if y[m].sum() < 2: continue
            if kind == "logreg":
                sc = StandardScaler().fit(Ztr[m]); clf = LogisticRegression(C=0.1, max_iter=2000, class_weight="balanced")
                clf.fit(sc.transform(Ztr[m]), y[m]); pr = lambda Z: clf.predict_proba(sc.transform(Z))[:, 1]
            else:
                clf = lgb.LGBMClassifier(n_estimators=50 if SMOKE else 2000, learning_rate=0.03, num_leaves=31,
                                         min_child_samples=50, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                                         scale_pos_weight=float((1 - y[m].mean()) / max(y[m].mean(), 1e-6)) ** 0.5,
                                         random_state=seed, verbose=-1)
                ev = [(Zva[mv], yv[mv])] if yv[mv].sum() > 0 else None
                clf.fit(Ztr[m], y[m], eval_set=ev, eval_metric="average_precision",
                        callbacks=[lgb.early_stopping(100, verbose=False)] if ev else None)
                pr = lambda Z: clf.predict_proba(Z)[:, 1]
            for Z, out in zip((Zva, Zte), outs): out[..., j] = pr(Z).reshape(out.shape[:2])
        return tuple(outs)
    return cached(f"{kind}_{'cross' if cross else 'own'}_fold{f}_N{N}", run)

for f in FOLD_IDS:
    SCORES[f]["Persistence (recent rise)"] = [(persistence(f, "val"), persistence(f, "test"))]
    for kind, nm in (("logreg", "Logistic"), ("lgbm", "LightGBM")):
        for cross in (False, True):
            t0 = time.time()
            SCORES[f][f"{nm} ({'own + cross' if cross else 'own'})"] = [tabular_model(f, kind, cross)]
            print(f"fold {f} {nm} cross={cross}: {time.time()-t0:.0f}s")

In [ ]:
# ==== 5.1 DEEP MODELS: single-task transformer and MT-CPTA (+ ablations) ====
class ShockNet(nn.Module):
    def __init__(self, cross=True, heads=("price", "shock", "vol"), d=64, layers=2):
        super().__init__()
        self.cross, self.heads = cross, heads
        self.enc = PatchEncoder(L, N_CH, d); self.cat = nn.Embedding(len(CATS), d)
        self.register_buffer("catv", torch.as_tensor(CAT, dtype=torch.long))
        if cross:
            el = nn.TransformerEncoderLayer(d, 4, 2 * d, 0.1, batch_first=True, norm_first=True)
            self.xattn = nn.TransformerEncoder(el, layers, enable_nested_tensor=False)
        self.h_shock = Head(d, len(HS)); self.h_price = Head(d, H) if "price" in heads else None
        self.h_vol = Head(d, 1) if "vol" in heads else None
    def forward(self, x):                                                   # (B, N, L, C)
        B = x.shape[0]
        h = self.enc(x.reshape(B * N, L, N_CH)).view(B, N, -1) + self.cat(self.catv)[None]
        if self.cross: h = self.xattn(h)
        out = dict(shock=self.h_shock(h))
        if self.h_price is not None: out["price"] = self.h_price(h)
        if self.h_vol is not None: out["vol"] = self.h_vol(h)[..., 0]
        return out

def focal(logits, y, m, gamma=2.0, alpha=0.75):
    p = torch.sigmoid(logits.float()); y = y.float(); m = m.float()
    ce = Fnn.binary_cross_entropy_with_logits(logits.float(), y, reduction="none")
    pt = p * y + (1 - p) * (1 - y); a = alpha * y + (1 - alpha) * (1 - y)
    return (a * (1 - pt) ** gamma * ce * m).sum() / m.sum().clamp(min=1)

def bce(logits, y, m, pos_weight=10.0):
    l = Fnn.binary_cross_entropy_with_logits(logits.float(), y.float(), reduction="none",
                                             pos_weight=torch.tensor(pos_weight, device=logits.device))
    return (l * m.float()).sum() / m.float().sum().clamp(min=1)

def make_loss(loss_kind="focal", lam=(1.0, 1.0, 0.5)):
    def lf(model, b):
        o = model(b["x"])
        L_ = lam[1] * (focal if loss_kind == "focal" else bce)(o["shock"], b["s"], b["sm"])
        if "price" in o: L_ = L_ + lam[0] * masked_huber(o["price"], b["y"], b["m"])
        if "vol" in o: L_ = L_ + lam[2] * Fnn.mse_loss(o["vol"].float(), b["v"])
        return L_
    return lf

def run_deep(name, build, f, seed, loss_kind="focal", bs=4):
    key = f"{name}__f{f}__s{seed}"; pf = PRED_DIR / f"{key}.npz"
    if pf.exists():
        z = np.load(pf); return z["val"].astype(np.float32), z["test"].astype(np.float32), z["price"].astype(np.float32)
    set_seed(seed); model = build(); rng = np.random.default_rng(seed)
    ex = {p: dict(s=LAB[f][p][0], sm=LAB[f][p][1], v=LAB[f][p][2]) for p in ("train", "val", "test")}
    tr, va, te = D[f]["train"], D[f]["val"], D[f]["test"]
    model, _ = fit(model, lambda ep: iterate_snap(*tr[:3], bs, True, rng, ex["train"]),
                   lambda: iterate_snap(*va[:3], bs, False, extra=ex["val"]), make_loss(loss_kind),
                   CKPT_DIR / f"{key}.pt", EPOCHS, 1e-3, patience=6, verbose=False)
    sig = lambda m, b: torch.sigmoid(m(b["x"])["shock"].float())
    pv = predict(model, iterate_snap(*va[:3], bs, False), fwd=sig); pt = predict(model, iterate_snap(*te[:3], bs, False), fwd=sig)
    pp = predict(model, iterate_snap(*te[:3], bs, False), fwd=lambda m, b: m(b["x"]).get("price", torch.zeros(b["y"].shape, device=DEVICE)).float())
    np.savez_compressed(pf, val=pv.astype(np.float16), test=pt.astype(np.float16), price=pp.astype(np.float16))
    return pv, pt, pp

DEEP = {"Transformer (own, single-task)": (lambda: ShockNet(cross=False, heads=("shock",)), "focal"),
        "MT-CPTA (proposed)": (lambda: ShockNet(cross=True), "focal"),
        "- cross-product attention": (lambda: ShockNet(cross=False), "focal"),
        "- multi-task (shock head only)": (lambda: ShockNet(cross=True, heads=("shock",)), "focal"),
        "- volatility head": (lambda: ShockNet(cross=True, heads=("price", "shock")), "focal"),
        "- focal loss (weighted BCE)": (lambda: ShockNet(cross=True), "bce")}
PRICE = {f: {} for f in FOLD_IDS}
for name, (build, lk) in DEEP.items():
    for f in FOLD_IDS:
        t0 = time.time(); outs = [run_deep(name, build, f, s, lk) for s in SEEDS]
        SCORES[f][name] = [(o[0], o[1]) for o in outs]; PRICE[f][name] = [o[2] for o in outs]
        print(f"{name} fold {f}: {time.time()-t0:.0f}s")

## 6. Evaluation

In [ ]:
# ==== 6. METRICS (raw + Platt-calibrated on validation; threshold from validation) ====
def ece(p, y, bins=10):
    e, edges = 0.0, np.linspace(0, 1, bins + 1)
    for a, b in zip(edges[:-1], edges[1:]):
        m = (p >= a) & (p < b) if b < 1 else (p >= a)
        if m.any(): e += m.mean() * abs(p[m].mean() - y[m].mean())
    return e

def platt(sv, yv):
    lr = LogisticRegression(C=1e4).fit(np.log(np.clip(sv, 1e-6, None))[:, None] if sv.min() >= 0 else sv[:, None], yv)
    return lambda s: lr.predict_proba(np.log(np.clip(s, 1e-6, None))[:, None] if sv.min() >= 0 else s[:, None])[:, 1]

def metrics_for(model, j):
    """Pooled over folds and averaged over seeds -> dict of metrics for horizon index j."""
    res = []
    for s in range(len(SCORES[FOLD_IDS[0]][model])):
        ys, ps, pc, pv_all, yv_all = [], [], [], [], []
        thr_list = []
        for f in FOLD_IDS:
            sv, st = SCORES[f][model][s]
            yv, mv = LAB[f]["val"][0][..., j], LAB[f]["val"][1][..., j]
            yt, mt = LAB[f]["test"][0][..., j], LAB[f]["test"][1][..., j]
            a, b = sv[..., j][mv], yv[mv]
            cal = platt(a, b) if 0 < b.sum() < len(b) else (lambda x: x)
            ca = cal(a)
            if 0 < b.sum():
                pr, rc, th = precision_recall_curve(b, ca); f1 = 2 * pr * rc / np.maximum(pr + rc, 1e-9)
                thr = th[np.argmax(f1[:-1])] if len(th) else 0.5
            else:
                thr = 0.5
            ys.append(yt[mt]); ps.append(st[..., j][mt]); pc.append(cal(st[..., j][mt])); thr_list.append(np.full(mt.sum(), thr))
        y, p, c, thr = map(np.concatenate, (ys, ps, pc, thr_list))
        yhat = c >= thr; tp = (yhat & (y > 0)).sum()
        prec = tp / max(yhat.sum(), 1); rec = tp / max((y > 0).sum(), 1)
        res.append(dict(PR_AUC=average_precision_score(y, p) if y.sum() else np.nan,
                        ROC_AUC=roc_auc_score(y, p) if 0 < y.sum() < len(y) else np.nan,
                        Brier_cal=brier_score_loss(y, c), ECE_raw=ece(np.clip(p, 0, 1), y) if p.max() <= 1 else np.nan,
                        ECE_cal=ece(c, y), Precision=prec, Recall=rec, F1=2 * prec * rec / max(prec + rec, 1e-9),
                        base_rate=y.mean()))
    return pd.DataFrame(res).mean(), pd.DataFrame(res).std()

MODELS = list(SCORES[FOLD_IDS[0]].keys())
rows = []
for j, h in enumerate(HS):
    for m in MODELS:
        mu, sd = metrics_for(m, j)
        rows.append(dict(h=h, model=m, **mu.to_dict(), PR_AUC_sd=sd.get("PR_AUC", np.nan)))
MET = pd.DataFrame(rows)
MAIN_MODELS = [m for m in MODELS if not m.startswith("- ")]
for h in HS:
    t = MET[(MET.h == h) & MET.model.isin(MAIN_MODELS)].drop(columns="h")
    save_table(t, f"tab_shock_h{h}", f"Shock detection at h = {h} days (test, pooled over folds, mean over seeds). "
               "Calibrated metrics use Platt scaling fitted on validation; the decision threshold maximises validation F1.",
               nd=3, bold_max=["PR_AUC", "ROC_AUC", "F1", "Precision", "Recall"], bold_min=["Brier_cal", "ECE_cal"])
abl = MET[MET.model.isin(["MT-CPTA (proposed)"] + [m for m in MODELS if m.startswith("- ")])].pivot(index="model", columns="h", values="PR_AUC")
abl.columns = [f"PR_AUC@{h}" for h in abl.columns]
save_table(abl.reset_index(), "tab_ablation", "MT-CPTA ablations: PR-AUC by horizon.", nd=4, bold_max=list(abl.columns))

In [ ]:
# ==== 6.1 SIGNIFICANCE: product-cluster bootstrap of PR-AUC differences =====
def pooled_scores(model, j, s=0):
    ys, ps, pid = [], [], []
    for f in FOLD_IDS:
        st = np.mean([x[1] for x in SCORES[f][model]], 0)
        yt, mt = LAB[f]["test"][0][..., j], LAB[f]["test"][1][..., j]
        ys.append(yt[mt]); ps.append(st[..., j][mt]); pid.append(np.broadcast_to(np.arange(N)[None], mt.shape)[mt])
    return map(np.concatenate, (ys, ps, pid))

B = 50 if SMOKE else 500
rows = []
for j, h in enumerate(HS):
    y, p1, pid = pooled_scores("MT-CPTA (proposed)", j)
    groups = [np.where(pid == i)[0] for i in range(N)]
    rng = np.random.default_rng(0); samples = [np.concatenate([groups[i] for i in rng.integers(0, N, N)]) for _ in range(B)]
    blk = []
    for m in MAIN_MODELS:
        if m == "MT-CPTA (proposed)": continue
        _, p2, _ = pooled_scores(m, j)
        d = [average_precision_score(y[s_], p1[s_]) - average_precision_score(y[s_], p2[s_]) for s_ in samples if y[s_].sum() > 0]
        obs = average_precision_score(y, p1) - average_precision_score(y, p2) if y.sum() else np.nan
        blk.append(dict(h=h, vs=m, d_PR_AUC=obs, CI_lo=np.quantile(d, 0.025) if d else np.nan, CI_hi=np.quantile(d, 0.975) if d else np.nan,
                        p_boot=2 * min(np.mean(np.array(d) <= 0), np.mean(np.array(d) >= 0)) if d else np.nan))
    blk = pd.DataFrame(blk); blk["p_holm"] = holm(blk.p_boot); rows.append(blk)
save_table(pd.concat(rows, ignore_index=True), "tab_significance",
           "MT-CPTA minus baseline PR-AUC with product-cluster bootstrap 95% CI and two-sided bootstrap p (Holm).", nd=4)

In [ ]:
# ==== 6.2 FIGURES: PR curves + reliability diagram ==========================
show = ["Persistence (recent rise)", "Logistic (own + cross)", "LightGBM (own)", "LightGBM (own + cross)",
        "Transformer (own, single-task)", "MT-CPTA (proposed)"]
fig, ax = plt.subplots(1, len(HS), figsize=(COL2, 2.4), sharey=True)
for j, h in enumerate(HS):
    for c, m in enumerate(show):
        y, p, _ = pooled_scores(m, j)
        if y.sum() == 0: continue
        pr, rc, _ = precision_recall_curve(y, p)
        ax[j].plot(rc, pr, color=PALETTE[c], lw=2.0 if "proposed" in m else 1.2, label=m)
    ax[j].axhline(y.mean(), color=GREY, ls="--", lw=0.8); ax[j].set_xlabel("Recall"); ax[j].set_title(f"h = {h} day" + ("s" if h > 1 else ""))
ax[0].set_ylabel("Precision"); ax[-1].legend(fontsize=6, loc="upper right"); fig.tight_layout(); savefig(fig, "fig_pr_curves")

j = HS.index(7)
fig, ax = plt.subplots(figsize=(COL1, 2.6))
for c, (m, lab_) in enumerate([("LightGBM (own + cross)", "LightGBM (own+cross), cal."), ("MT-CPTA (proposed)", "MT-CPTA, cal.")]):
    ys, cs = [], []
    for f in FOLD_IDS:
        sv, st = np.mean([x[0] for x in SCORES[f][m]], 0), np.mean([x[1] for x in SCORES[f][m]], 0)
        yv, mv = LAB[f]["val"][0][..., j], LAB[f]["val"][1][..., j]; yt, mt = LAB[f]["test"][0][..., j], LAB[f]["test"][1][..., j]
        cal = platt(sv[..., j][mv], yv[mv]) if 0 < yv[mv].sum() < mv.sum() else (lambda x: x)
        ys.append(yt[mt]); cs.append(cal(st[..., j][mt]))
    y, cc = np.concatenate(ys), np.concatenate(cs)
    qs = np.unique(np.quantile(cc, np.linspace(0, 1, 11)))
    bi = np.clip(np.digitize(cc, qs[1:-1]), 0, len(qs) - 2)
    xm = [cc[bi == b].mean() for b in range(len(qs) - 1) if (bi == b).any()]
    ym = [y[bi == b].mean() for b in range(len(qs) - 1) if (bi == b).any()]
    ax.plot(xm, ym, marker="o", color=PALETTE[c + 3], label=lab_)
lim = max(ax.get_xlim()[1], ax.get_ylim()[1]); ax.plot([0, lim], [0, lim], color=GREY, ls="--", lw=0.8)
ax.set_xlabel("Predicted probability"); ax.set_ylabel("Observed frequency"); ax.set_title("Reliability (h = 7, quantile bins)")
ax.legend(fontsize=6); fig.tight_layout(); savefig(fig, "fig_reliability")

## 7. Lead-time analysis
A *shock event* is a day d with a 1-day increase above theta(c, 1) and no other such increase in the previous
14 days, inside a test window. For the h = 14 detector, an alarm at origin t (d - 14 <= t < d) counts as a warning
if the calibrated score exceeds the validation-optimal threshold. Lead time = d - (earliest warning origin).

In [ ]:
# ==== 7. LEAD TIMES =========================================================
j14 = HS.index(14)
def lead_times(model):
    leads, n_events = [], 0
    for f in FOLD_IDS:
        sv, st = np.mean([x[0] for x in SCORES[f][model]], 0), np.mean([x[1] for x in SCORES[f][model]], 0)
        yv, mv = LAB[f]["val"][0][..., j14], LAB[f]["val"][1][..., j14]
        cal = platt(sv[..., j14][mv], yv[mv]) if 0 < yv[mv].sum() < mv.sum() else (lambda x: x)
        ca = cal(sv[..., j14][mv])
        pr, rc, th = precision_recall_curve(yv[mv], ca) if yv[mv].sum() else (None, None, [0.5])
        thr = th[np.argmax((2 * pr * rc / np.maximum(pr + rc, 1e-9))[:-1])] if yv[mv].sum() and len(th) else 0.5
        O = D[f]["test"][3][:, 0, 1]; o_pos = {o: k for k, o in enumerate(O)}
        lp = prep["logP"][IDX]; th1 = TH[f][CAT, 0]
        for i in range(N):
            r1 = 100 * np.diff(lp[i]); last = -99
            for d in range(FOLDS[f]["test_start"] + 1, FOLDS[f]["test_end"] + 1):
                if r1[d - 1] >= th1[i] and not prep["cf_day"][d]:
                    if d - last > 14:
                        n_events += 1
                        warn = [d - o for o in range(d - 14, d) if o in o_pos and cal(st[o_pos[o], i, j14:j14 + 1])[0] >= thr]
                        leads.append(max(warn) if warn else 0)
                    last = d
    return np.array(leads), n_events

LT = {m: lead_times(m) for m in ["LightGBM (own + cross)", "Transformer (own, single-task)", "MT-CPTA (proposed)"]}
rows = [dict(model=m, events=n, detected=(l > 0).mean() if n else np.nan, median_lead_days=np.median(l[l > 0]) if (l > 0).any() else np.nan,
             detected_7d_ahead=(l >= 7).mean() if n else np.nan) for m, (l, n) in LT.items()]
save_table(pd.DataFrame(rows), "tab_lead_time", "Early-warning performance for 1-day shock events (h = 14 detector).", nd=3)
fig, ax = plt.subplots(figsize=(COL1, 2.4))
for c, (m, (l, n)) in enumerate(LT.items()):
    if n == 0: continue
    ks = np.arange(1, 15); ax.plot(ks, [(l >= k).mean() for k in ks], marker="o", ms=3, color=PALETTE[c + 3 if c == 2 else c], label=m)
ax.set_xlabel("Warning at least k days ahead"); ax.set_ylabel("Share of shock events"); ax.legend(fontsize=6)
ax.set_title("Early-warning curve"); fig.tight_layout(); savefig(fig, "fig_lead_time")

## 8. Event study around platform batch-update days, and category breakdown

In [ ]:
# ==== 8. EVENT STUDY + CATEGORY PR-AUC ======================================
j7 = HS.index(7)
rows = []
for f in FOLD_IDS:
    O = np.concatenate([D[f]["val"][3][:, 0, 1], D[f]["test"][3][:, 0, 1]])
    S_all = np.concatenate([np.mean([x[0] for x in SCORES[f]["MT-CPTA (proposed)"]], 0), np.mean([x[1] for x in SCORES[f]["MT-CPTA (proposed)"]], 0)])
    Yl = np.concatenate([LAB[f]["val"][0], LAB[f]["test"][0]]); Ml = np.concatenate([LAB[f]["val"][1], LAB[f]["test"][1]])
    for bd in np.where(prep["batch_day"])[0]:
        for k, o in enumerate(O):
            rel = o - bd
            if -14 <= rel <= 7:
                m = Ml[k, :, j7]
                rows.append(dict(fold=f, rel=rel, score=S_all[k, :, j7][m].mean(), rate=Yl[k, :, j7][m].mean()))
ES = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(COL1, 2.4))
if len(ES):
    g = ES.groupby("rel").mean(numeric_only=True)
    ax.plot(g.index, g.score, color=PALETTE[6], marker="o", ms=3, label="mean predicted score (MT-CPTA)")
    ax.plot(g.index, g.rate, color=GREY, ls="--", label="realised 7-day shock rate")
    ax.axvline(0, color=PALETTE[1], lw=0.8)
else:
    ax.text(0.5, 0.5, "no batch-update day inside val/test windows", ha="center", transform=ax.transAxes)
ax.set_xlabel("Days relative to batch-update day"); ax.legend(fontsize=6); ax.set_title("Event study (h = 7)")
fig.tight_layout(); savefig(fig, "fig_event_study")

rows = []
for c in np.unique(CAT):
    r = dict(category=CATS[c], n_products=int((CAT == c).sum()))
    for m in ["LightGBM (own + cross)", "Transformer (own, single-task)", "MT-CPTA (proposed)"]:
        y, p, pid = pooled_scores(m, j7); sel = CAT[pid] == c
        r[m] = average_precision_score(y[sel], p[sel]) if y[sel].sum() > 0 else np.nan
        r["base_rate"] = y[sel].mean()
    rows.append(r)
save_table(pd.DataFrame(rows).sort_values("n_products", ascending=False), "tab_prauc_by_category", "PR-AUC at h = 7 by category.", nd=3)

In [ ]:
# ==== 8.1 SIDE BENEFIT: accuracy of MT-CPTA's price head vs Naive ===========
rows = []
for m in ["MT-CPTA (proposed)", "- cross-product attention"]:
    for f in FOLD_IDS:
        te = D[f]["test"]; s1, _ = evaluate(np.mean(PRICE[f][m], 0), te[1], te[2], te[3], prep, FOLDS[f])
        s0, _ = evaluate(np.zeros_like(te[1]), te[1], te[2], te[3], prep, FOLDS[f])
        rows.append(dict(model=m, fold=f, sMAPE=s1.sMAPE.mean(), sMAPE_naive=s0.sMAPE.mean(), CC_sMAPE=s1.CC_sMAPE.mean(), CC_sMAPE_naive=s0.CC_sMAPE.mean()))
save_table(pd.DataFrame(rows).groupby("model").mean(numeric_only=True).drop(columns="fold").reset_index(), "tab_price_head",
           "Price-path head of the multi-task model vs the Naive forecast (test, mean over folds).", nd=3)

In [ ]:
# ==== 9. ASSET INDEX + RESULTS SUMMARY ======================================
lines = [f"# Paper 4 - results summary ({time.strftime('%Y-%m-%d')})", f"* {N} active food SKUs; folds {FOLD_IDS}; seeds {SEEDS}."]
for h in HS:
    t = MET[(MET.h == h) & MET.model.isin(MAIN_MODELS)].set_index("model")
    lines.append(f"* h={h}: base rate {t.base_rate.iloc[0]:.4f}; best PR-AUC {t.PR_AUC.idxmax()} ({t.PR_AUC.max():.3f}); MT-CPTA {t.loc['MT-CPTA (proposed)', 'PR_AUC']:.3f}")
(PAPER_DIR / "RESULTS_SUMMARY.md").write_text("\n".join(lines)); print("\n".join(lines))
print("figures:", sorted(p.name for p in FIG_DIR.glob("*.pdf"))); print("tables:", sorted(p.name for p in TAB_DIR.glob("*.tex")))